# tau3 Multi-objective vs Single-objective GRPO (Colab Free T4)

Runs the pure-RL loop with **Qwen2.5-3B-Instruct** (QLoRA, custom agentic GRPO)
on the synthesized tau3 retail suite, for two arms:

- `single` — reward = correctness only
- `multi` — reward = weighted sum (correctness 1.0, efficiency 0.3, tool_safety 0.5)

Select **Runtime > Change runtime type > T4 GPU** before running.

> Environment note: all code runs in the **project venv** via `uv run` (torch,
> unsloth, and `harnessx` live there), so notebook cells must NOT `import`
> `experiments`/`harnessx` directly — the kernel uses a different Python.

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')

# project location
PROJECT = '/content/agentic'
# move checkpoints off the ephemeral disk so they survive session timeouts
os.environ.setdefault('CKPT', '/content/drive/MyDrive/agentic_tau3_rl')
print('Drive mounted, CKPT =', os.environ['CKPT'])

In [ ]:
# install uv, clone repo, checkout experiment branch, sync deps
!pip -q install uv
if not os.path.isdir(PROJECT):
    !git clone https://github.com/culey24/agentic.git {PROJECT}
%cd {PROJECT}
!git switch experiment/multi-objective-rl
!uv sync --dev
branch = !git branch --show-current
print('repo ready, branch =', branch)

In [ ]:
# GPU stack: torch + unsloth, installed INTO the project venv (uv run uses it)
VENV = '/content/agentic/.venv/bin/python'
!uv pip install --python {VENV} torch --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python {VENV} unsloth xformers trl peft bitsandbytes
print('GPU stack ready in venv')

In [ ]:
# sanity: load the model + dry rollout BEFORE training.
# Runs via uv run so everything uses the project venv (torch/unsloth/harnessx).
!uv run python experiments/rl/colab/dry_run.py --task 0 --max-turns 20 2>&1 | tail -30

In [ ]:
# Arm 1: SINGLE objective (correctness only)
ckpt = os.path.join(os.environ['CKPT'], 'single')
!uv run python experiments/rl/colab/run.py \
    --arm single \
    --rounds 6 --rollouts 8 --concurrency 4 --max-turns 200 \
    --lr 5e-5 --kl-beta 0.04 --clip-ratio 0.2 \
    --out {ckpt} --save-lora {ckpt}/lora \
    --model Qwen/Qwen2.5-3B-Instruct 2>&1 | tee {ckpt}.log

In [ ]:
# Arm 2: MULTI objective (weighted sum)
ckpt = os.path.join(os.environ['CKPT'], 'multi')
!uv run python experiments/rl/colab/run.py \
    --arm multi --weights correctness=1.0,efficiency=0.3,tool_safety=0.5 \
    --rounds 6 --rollouts 8 --concurrency 4 --max-turns 200 \
    --lr 5e-5 --kl-beta 0.04 --clip-ratio 0.2 \
    --out {ckpt} --save-lora {ckpt}/lora \
    --model Qwen/Qwen2.5-3B-Instruct 2>&1 | tee {ckpt}.log

In [ ]:
# Compare the two arms: pass@N curves + objective means + Pareto
import json
import os

import matplotlib.pyplot as plt


def load_rounds(arm):
    p = os.path.join(os.environ['CKPT'], arm, 'rounds.jsonl')
    if not os.path.exists(p):
        return []
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

single, multi = load_rounds('single'), load_rounds('multi')
print(f"{'round':<6}{'single pass@N':<16}{'multi pass@N':<16}{'single reward':<16}{'multi reward':<16}")
for i in range(max(len(single), len(multi))):
    s = single[i] if i < len(single) else {}
    m = multi[i] if i < len(multi) else {}
    print(f"R{i:<5}{s.get('pass_rate', float('nan')):<16.3f}{m.get('pass_rate', float('nan')):<16.3f}"
          f"{s.get('mean_reward', float('nan')):<16.3f}{m.get('mean_reward', float('nan')):<16.3f}")

plt.figure(figsize=(6, 4))
plt.plot([r['pass_rate'] for r in single], '-o', label='single')
plt.plot([r['pass_rate'] for r in multi], '-s', label='multi')
plt.xlabel('round'); plt.ylabel('pass@N'); plt.title('tau3 GRPO: single vs multi objective')
plt.legend(); plt.grid(alpha=.3); plt.show()

def means(arm):
    p = os.path.join(os.environ['CKPT'], arm, 'rollouts.jsonl')
    if not os.path.exists(p):
        return {}
    with open(p) as f:
        vecs = [json.loads(l)['rewards'] for l in f if l.strip()]
    names = sorted({n for v in vecs for n in v})
    return {n: sum(v.get(n, 0) for v in vecs)/len(vecs) for n in names} if vecs else {}

print('objective means  single:', {k: round(v, 3) for k, v in means('single').items()})
print('objective means  multi :', {k: round(v, 3) for k, v in means('multi').items()})

### Interpretation
- If `multi` keeps `correctness` ≈ `single` while raising `efficiency`/`tool_safety`, the
  multi-objective reward shaping is **free**: it improves secondary objectives without
  sacrificing the primary one (verify no correctness regression in pass@N).
- If `single` regresses `efficiency`/`tool_safety` to win correctness, that is the expected
  single-objective overfit — evidence for why multi-objective matters.
- Checkpoints live under your Drive `CKPT` dir (`rollouts.jsonl`, `rounds.jsonl`, `lora/`).
